In [2]:
# ============================================================================
# KAGGLE NOTEBOOK - TRANSLATION BENCHMARK WITH OLLAMA
# ============================================================================

# CELL 1: Install and Start Ollama
# ============================================================================

import subprocess
import time
import os

print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\nStarting Ollama service...")
# Start Ollama in background
proc = subprocess.Popen(['ollama', 'serve'], 
                       stdout=subprocess.PIPE, 
                       stderr=subprocess.PIPE)
time.sleep(10)  # Wait for service to start

print("✅ Ollama service started!")

Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%                                                         6.3%                                                                  7.9%#####################################                            64.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Starting Ollama service...
✅ Ollama service started!


In [9]:
# CELL: Install Ollama Python Package
!pip install ollama

print("✅ Ollama Python package installed!")

✅ Ollama Python package installed!


In [3]:
# CELL: Test GPU Detection
import subprocess

# Check if GPU is available
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# This should show your T4 GPU!


# Expected output:
'''
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.xx.xx    Driver Version: 525.xx.xx    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
+-------------------------------+----------------------+----------------------+'''

Fri Nov 14 16:19:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

'\n+-----------------------------------------------------------------------------+\n| NVIDIA-SMI 525.xx.xx    Driver Version: 525.xx.xx    CUDA Version: 12.0     |\n|-------------------------------+----------------------+----------------------+\n| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |\n|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |\n+-------------------------------+----------------------+----------------------+'

In [4]:
# ============================================================================
# CELL 2: Pull Models
# ============================================================================

print("Pulling models (this takes a few minutes)...")
print("\n1/3 Pulling qwen2:1.5b...")
!ollama pull qwen2:1.5b

print("\n2/3 Pulling llama3.2:1b...")
!ollama pull llama3.2:1b

print("\n3/3 Pulling gemma2:2b...")
!ollama pull gemma2:2b

print("\n✅ All models downloaded!")
!ollama list

Pulling models (this takes a few minutes)...

1/3 Pulling qwen2:1.5b...
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 405b56374e02:   0% ▕                  ▏ 444 KB/934 MB                  pulling manifest 
pulling 405b56374e02:   2% ▕                  ▏  16 MB/934 MB                  pulling manifest 
pulling 405b56374e02:   7% ▕█                 ▏  62 MB/934 MB                  pulling manifest 
pulling 405b56374e02:  12% ▕██                ▏ 113 MB/934 MB                  pulling manifest 
pulling 405b56374e02:  15% ▕██                ▏ 143 MB/934 MB                  pulling manifest 
pulling 405b56374e02:  20% ▕███               ▏ 187 MB/934 MB                  pulling manifest 
pulling 405b56374e02:  25% ▕████              ▏ 229 MB/934 MB                  pulling manifest 
pulling 405b56374e02:  27% ▕████              ▏ 252 MB/934 MB                  

In [5]:
# ============================================================================
# CELL 3: Import Libraries and Define Functions
# ============================================================================

import pandas as pd
import re
import sys
from datetime import datetime

# Configuration
TEST_MODE = True  # Set to False for full dataset
TEST_ROWS = 50
MODELS_TO_TEST = ["qwen2:1.5b", "llama3.2:1b", "gemma2:2b"]
BATCH_SIZE = 10
CHECKPOINT_FREQ = 50

# Text processing functions
def fix_encoding(text):
    """Fix common encoding issues"""
    if pd.isna(text) or str(text).strip() == '':
        return ''
    
    text = str(text)
    
    encoding_fixes = {
        'Ã©': 'é', 'Ã¨': 'è', 'Ãª': 'ê', 'Ã«': 'ë',
        'Ã ': 'à', 'Ã¢': 'â', 'Ã§': 'ç',
        'Ã´': 'ô', 'Ã¹': 'ù', 'Ã»': 'û',
        'Ã¯': 'ï', 'Ã®': 'î',
        'Å"': 'œ', 'Ã†': 'Æ',
        'â€™': "'", 'â€œ': '"', 'â€': '"',
        'â€"': '-', 'â€"': '-',
        'Â': '', 'Ã': ''
    }
    
    for wrong, correct in encoding_fixes.items():
        text = text.replace(wrong, correct)
    
    return text

def clean_text(text):
    """Clean text by removing special characters, extra spaces, etc."""
    if pd.isna(text) or str(text).strip() == '':
        return ''
    
    text = fix_encoding(text)
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\-àâäéèêëïîôùûüÿçÀÂÄÉÈÊËÏÎÔÙÛÜŸÇ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def concatenate_text(designation, description):
    """Concatenate designation and description with proper formatting"""
    desig = str(designation) if pd.notna(designation) else ''
    desc = str(description) if pd.notna(description) else ''
    
    if desig and desc:
        combined = f"{desig}. {desc}"
    elif desig:
        combined = desig
    elif desc:
        combined = desc
    else:
        combined = ''
    
    return combined.strip('. ')

# Translation functions
def translate_single(text, model_name):
    """Translate a single text"""
    if not text or text.strip() == '':
        return '[EMPTY]'
    
    prompt = f"""Translate the following French text to English. Provide ONLY the English translation in a single paragraph without line breaks:

{text}

English translation:"""
    
    try:
        import ollama
        response = ollama.generate(model=model_name, prompt=prompt, stream=False)
        translation = response['response'].strip()
        
        # Remove common prefixes
        translation = re.sub(r'^(Translation:|English:|Here is the translation:)\s*', '', translation, flags=re.IGNORECASE)
        
        # Remove all newlines to prevent CSV corruption
        translation = translation.replace('\n', ' ').replace('\r', ' ')
        translation = re.sub(r'\s+', ' ', translation).strip()
        
        return translation if translation else text
    except Exception as e:
        print(f"  ⚠️  Error translating with {model_name}: {e}")
        return text

def translate_batch(texts, model_name):
    """Translate a batch of texts"""
    results = []
    for text in texts:
        translation = translate_single(text, model_name)
        results.append(translation)
    return results

print("✅ Functions defined!")

✅ Functions defined!


In [6]:
# ============================================================================
# CELL 4: Load Data
# ============================================================================

# IMPORTANT: Upload your X_train.csv using Kaggle's "Add data" button first!
# It will be available at: /kaggle/input/your-dataset-name/X_train.csv

# Update this path to match your uploaded dataset:
INPUT_PATH = '/kaggle/input/x-train-update-csv/X_train_update.csv'  # Change this!
OUTPUT_PATH = '/kaggle/working/X_train_translated_benchmark.csv'

print(f"Loading data from {INPUT_PATH}...")
df = pd.read_csv(INPUT_PATH)

if TEST_MODE:
    df = df.head(TEST_ROWS).copy()
    OUTPUT_PATH = OUTPUT_PATH.replace('.csv', '_test.csv')

print(f"✅ Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

Loading data from /kaggle/input/x-train-update-csv/X_train_update.csv...
✅ Loaded 50 rows
Columns: ['Unnamed: 0', 'designation', 'description', 'productid', 'imageid']


In [7]:
# ============================================================================
# CELL 5: Prepare Text Columns
# ============================================================================

print("\n📝 Creating original and cleaned text columns...")

# Create original concatenated text
if 'text_original' not in df.columns:
    df['text_original'] = df.apply(
        lambda row: concatenate_text(row.get('designation'), row.get('description')), 
        axis=1
    )
    print(f"✅ Created text_original column")

# Create cleaned text
if 'text_cleaned' not in df.columns:
    df['text_cleaned'] = df['text_original'].apply(clean_text)
    print(f"✅ Created text_cleaned column")

# Show sample
print("\n📋 Sample data:")
print(df[['text_original', 'text_cleaned']].head(2))


📝 Creating original and cleaned text columns...
✅ Created text_original column
✅ Created text_cleaned column

📋 Sample data:
                                       text_original  \
0  Olivia: Personalisiertes Notizbuch / 150 Seite...   
1  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...   

                                        text_cleaned  
0  Olivia: Personalisiertes Notizbuch 150 Seiten ...  
1  Journal Des Arts Le N 133 Du 28 09 2001 - L ar...  


In [ ]:
# CELL: Delete Previous Output
import os

OUTPUT_PATH = '/kaggle/working/X_train_translated_benchmark_test.csv'

if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)
    print(f"✅ Deleted previous output: {OUTPUT_PATH}")
else:
    print("No previous output found")

In [11]:
# CELL: Inspect DataFrame
print("Current columns:")
print(df.columns.tolist())

print("\nTranslation columns:")
trans_cols = [col for col in df.columns if 'text_translated_' in col]
for col in trans_cols:
    non_null = df[col].notna().sum()
    print(f"  {col}: {non_null}/{len(df)} rows translated")

print("\nSample data:")
print(df.head(2))

Current columns:
['Unnamed: 0', 'designation', 'description', 'productid', 'imageid', 'text_original', 'text_cleaned', 'text_translated_qwen2_1_5b', 'text_translated_llama3_2_1b', 'text_translated_gemma2_2b']

Translation columns:
  text_translated_qwen2_1_5b: 50/50 rows translated
  text_translated_llama3_2_1b: 50/50 rows translated
  text_translated_gemma2_2b: 50/50 rows translated

Sample data:
   Unnamed: 0                                        designation description  \
0           0  Olivia: Personalisiertes Notizbuch / 150 Seite...         NaN   
1           1  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...         NaN   

    productid     imageid                                      text_original  \
0  3804725264  1263597046  Olivia: Personalisiertes Notizbuch / 150 Seite...   
1   436067568  1008141237  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...   

                                        text_cleaned  \
0  Olivia: Personalisiertes Notizbuch 150 Seiten ...   
1  Jo

In [12]:
# ============================================================================
# CELL 6: Benchmark Translation (FIXED VERSION)
# ============================================================================

print(f"\n{'='*80}")
print(f"BENCHMARKING {len(MODELS_TO_TEST)} MODELS")
print(f"{'='*80}")
print(f"Models: {', '.join(MODELS_TO_TEST)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total rows: {len(df)}")
print(f"{'='*80}\n")

benchmark_results = {}

for model_name in MODELS_TO_TEST:
    print(f"\n{'='*80}")
    print(f"🤖 TESTING MODEL: {model_name}")
    print(f"{'='*80}")
    
    col_name = f'text_translated_{model_name.replace(":", "_").replace(".", "_")}'
    
    # Initialize column if it doesn't exist
    if col_name not in df.columns:
        df[col_name] = None
        start_idx = 0
        print(f"Starting fresh translation...")
    else:
        # Count how many are actually translated (not empty, not same as original)
        already_done = 0
        for idx in df.index:
            trans = df.loc[idx, col_name]
            original = df.loc[idx, 'text_cleaned']
            # Check if translated AND different from original
            if pd.notna(trans) and trans != '[EMPTY]' and trans != original:
                already_done += 1
            else:
                break  # Stop at first untranslated row
        
        start_idx = already_done
        
        if already_done > 0:
            print(f"✅ Found {already_done} valid translations, resuming from row {start_idx}...")
        else:
            print(f"No valid translations found, starting fresh...")
    
    total_rows = len(df)
    model_start_time = time.time()
    
    # Process in batches
    for batch_start in range(start_idx, total_rows, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, total_rows)
        batch_df = df.iloc[batch_start:batch_end]
        
        # Get cleaned texts for this batch
        texts_to_translate = batch_df['text_cleaned'].tolist()
        
        # Translate batch
        print(f"  📦 Translating batch {batch_start+1}-{batch_end}...", end='', flush=True)
        batch_start_time = time.time()
        
        translations = translate_batch(texts_to_translate, model_name)
        
        batch_time = time.time() - batch_start_time
        
        # Update dataframe
        for i, idx in enumerate(batch_df.index):
            df.loc[idx, col_name] = translations[i]
        
        # Progress stats
        completed = batch_end
        elapsed = time.time() - model_start_time
        rate = (completed - start_idx) / elapsed if elapsed > 0 else 0
        remaining = total_rows - completed
        eta_seconds = remaining / rate if rate > 0 else 0
        
        print(f" ✅ {batch_time:.1f}s ({rate:.2f} rows/sec)")
        
        # Checkpoint save
        if completed % CHECKPOINT_FREQ < BATCH_SIZE or completed == total_rows:
            print(f"  💾 Saving checkpoint at row {completed}...")
            df.to_csv(OUTPUT_PATH, index=False, quoting=1)
    
    # Model summary
    total_time = time.time() - model_start_time
    avg_time_per_row = total_time / (total_rows - start_idx) if (total_rows - start_idx) > 0 else 0
    benchmark_results[model_name] = {
        'total_time': total_time,
        'avg_time_per_row': avg_time_per_row,
        'rows_per_second': 1 / avg_time_per_row if avg_time_per_row > 0 else 0
    }
    
    print(f"\n  ⏱️  Model Summary:")
    print(f"    Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"    Avg per row: {avg_time_per_row:.2f}s")
    print(f"    Throughput: {benchmark_results[model_name]['rows_per_second']:.2f} rows/sec")

print("\n✅ All models translated!")


BENCHMARKING 3 MODELS
Models: qwen2:1.5b, llama3.2:1b, gemma2:2b
Batch size: 10
Total rows: 50


🤖 TESTING MODEL: qwen2:1.5b
No valid translations found, starting fresh...
  📦 Translating batch 1-10... ✅ 12.6s (0.80 rows/sec)
  📦 Translating batch 11-20... ✅ 21.0s (0.60 rows/sec)
  📦 Translating batch 21-30... ✅ 11.4s (0.67 rows/sec)
  📦 Translating batch 31-40... ✅ 15.2s (0.66 rows/sec)
  📦 Translating batch 41-50... ✅ 10.5s (0.71 rows/sec)
  💾 Saving checkpoint at row 50...

  ⏱️  Model Summary:
    Total time: 70.7s (1.2 min)
    Avg per row: 1.41s
    Throughput: 0.71 rows/sec

🤖 TESTING MODEL: llama3.2:1b
No valid translations found, starting fresh...
  📦 Translating batch 1-10... ✅ 8.8s (1.14 rows/sec)
  📦 Translating batch 11-20... ✅ 10.9s (1.02 rows/sec)
  📦 Translating batch 21-30... ✅ 11.3s (0.97 rows/sec)
  📦 Translating batch 31-40... ✅ 10.7s (0.96 rows/sec)
  📦 Translating batch 41-50... ✅ 8.0s (1.01 rows/sec)
  💾 Saving checkpoint at row 50...

  ⏱️  Model Summary:
    T

In [ ]:
# ============================================================================
# CELL: Validate Translations
# ============================================================================

print("🔍 VALIDATING TRANSLATIONS...")
print("="*80)

for model in MODELS_TO_TEST:
    col_name = f'text_translated_{model.replace(":", "_").replace(".", "_")}'
    
    if col_name not in df.columns:
        print(f"❌ {model}: Column doesn't exist")
        continue
    
    # Check statistics
    total = len(df)
    not_null = df[col_name].notna().sum()
    empty = (df[col_name] == '[EMPTY]').sum()
    
    # Check how many are identical to original (NOT translated)
    identical = 0
    for idx in df.index:
        trans = df.loc[idx, col_name]
        original = df.loc[idx, 'text_cleaned']
        if pd.notna(trans) and trans == original:
            identical += 1
    
    actually_translated = not_null - empty - identical
    
    print(f"\n{model}:")
    print(f"  Total rows: {total}")
    print(f"  Non-null: {not_null}")
    print(f"  Empty: {empty}")
    print(f"  Identical to original (not translated): {identical}")
    print(f"  ✅ Actually translated: {actually_translated}")
    
    if identical > 0:
        print(f"  ⚠️  WARNING: {identical} rows appear untranslated!")

print("\n" + "="*80)

In [13]:
# ============================================================================
# CELL 7: Save Final Results and Show Comparison
# ============================================================================

print(f"\n💾 Saving final result to {OUTPUT_PATH}...")
df.to_csv(OUTPUT_PATH, index=False, quoting=1)  # Quote all fields

print("\n" + "=" * 80)
print("📊 BENCHMARK RESULTS COMPARISON")
print("=" * 80)

print(f"\n{'Model':<20} {'Total Time':<15} {'Avg/Row':<15} {'Rows/Sec':<15}")
print("-" * 80)

sorted_results = sorted(benchmark_results.items(), key=lambda x: x[1]['total_time'])

for model_name, stats in sorted_results:
    total_time = stats['total_time']
    avg_time = stats['avg_time_per_row']
    rows_per_sec = stats['rows_per_second']
    
    print(f"{model_name:<20} {total_time:>8.1f}s ({total_time/60:>5.1f}m) {avg_time:>8.2f}s      {rows_per_sec:>8.2f}")

# Estimate for full dataset
if TEST_MODE:
    print(f"\n📊 ESTIMATION FOR FULL DATASET (67,932 rows):")
    print("-" * 80)
    
    for model_name, stats in sorted_results:
        full_time_seconds = stats['avg_time_per_row'] * 67932
        full_time_hours = full_time_seconds / 3600
        full_time_days = full_time_hours / 24
        
        print(f"{model_name:<20} ~{full_time_hours:>6.1f} hours (~{full_time_days:>4.1f} days)")

# Show sample translations
print(f"\n{'='*80}")
print("📋 SAMPLE TRANSLATIONS (First 3 rows)")
print("=" * 80)

for idx in range(min(3, len(df))):
    print(f"\n{'─'*80}")
    print(f"ROW {idx}")
    print(f"{'─'*80}")
    
    row = df.iloc[idx]
    
    print(f"\n🔵 ORIGINAL:")
    print(f"  {row['text_original'][:200] if pd.notna(row.get('text_original')) else '[EMPTY]'}")
    
    print(f"\n🧹 CLEANED:")
    print(f"  {row['text_cleaned'][:200] if pd.notna(row.get('text_cleaned')) else '[EMPTY]'}")
    
    for model in MODELS_TO_TEST:
        col_name = f'text_translated_{model.replace(":", "_").replace(".", "_")}'
        if col_name in df.columns:
            print(f"\n🟢 {model.upper()}:")
            print(f"  {row[col_name][:200] if pd.notna(row.get(col_name)) else '[EMPTY]'}")

print("\n" + "=" * 80)
print("✅ BENCHMARK COMPLETE!")
print(f"Output saved to: {OUTPUT_PATH}")
print("=" * 80)


💾 Saving final result to /kaggle/working/X_train_translated_benchmark_test.csv...

📊 BENCHMARK RESULTS COMPARISON

Model                Total Time      Avg/Row         Rows/Sec       
--------------------------------------------------------------------------------
llama3.2:1b              49.7s (  0.8m)     0.99s          1.01
qwen2:1.5b               70.7s (  1.2m)     1.41s          0.71
gemma2:2b               105.3s (  1.8m)     2.11s          0.47

📊 ESTIMATION FOR FULL DATASET (67,932 rows):
--------------------------------------------------------------------------------
llama3.2:1b          ~  18.7 hours (~ 0.8 days)
qwen2:1.5b           ~  26.7 hours (~ 1.1 days)
gemma2:2b            ~  39.7 hours (~ 1.7 days)

📋 SAMPLE TRANSLATIONS (First 3 rows)

────────────────────────────────────────────────────────────────────────────────
ROW 0
────────────────────────────────────────────────────────────────────────────────

🔵 ORIGINAL:
  Olivia: Personalisiertes Notizbuch / 150 Seiten /

In [14]:
# ============================================================================
# CELL 8: Download Results
# ============================================================================

print("\n📥 Your translated file is ready to download!")
print(f"Location: {OUTPUT_PATH}")
print("\nTo download:")
print("1. Look in the right sidebar under 'Output'")
print("2. Click on the file to download")
print("\nOr run this cell to see the file:")

!ls -lh /kaggle/working/*.csv


📥 Your translated file is ready to download!
Location: /kaggle/working/X_train_translated_benchmark_test.csv

To download:
1. Look in the right sidebar under 'Output'
2. Click on the file to download

Or run this cell to see the file:
-rw-r--r-- 1 root root 159K Nov 14 16:43 /kaggle/working/X_train_translated_benchmark_test.csv
